# 关联规则挖掘：Apriori 算法实战

本 Notebook 是 [02_关联规则_Apriori.md](02_关联规则_Apriori.md) 的配套代码。我们将使用 Python 的 `mlxtend` 库来演示关联规则挖掘的过程。

## 目录

1. **环境准备**
2. **验证手算示例**：使用文档中“3.3 手算推演示例”的数据，验证我们的计算结果。
3. **综合案例分析**：使用更复杂的购物篮数据，演示完整的挖掘流程。
4. **FP-Growth 算法**：对比 Apriori 和 FP-Growth 的使用。

## 1. 环境准备

我们需要安装 `mlxtend` 库。如果尚未安装，请取消下方注释并运行。

In [10]:
# !pip install mlxtend pandas

In [11]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules, fpgrowth

## 2. 验证手算示例

为了验证我们在文档 **3.3 手算推演示例** 中的计算过程，我们构造完全相同的数据集进行代码测试。

**数据集回顾**：

| 事务 ID | 购买商品 |
| :--- | :--- |
| T1 | 尿布, 啤酒, 花生 |
| T2 | 尿布, 啤酒 |
| T3 | 啤酒, 花生 |
| T4 | 尿布, 花生 |
| T5 | 尿布, 啤酒, 花生 |
| T6 | 啤酒 |

**参数设定**：
- 最小支持度 (min_support) = 50% (3/6)
- 最小置信度 (min_threshold) = 0.0 (为了查看所有规则)

In [12]:
# 1. 构建手算示例数据集
dataset_manual = [
    ['尿布', '啤酒', '花生'],
    ['尿布', '啤酒'],
    ['啤酒', '花生'],
    ['尿布', '花生'],
    ['尿布', '啤酒', '花生'],
    ['啤酒']
]

# 2. 数据编码 (One-Hot)
te = TransactionEncoder()
te_ary = te.fit(dataset_manual).transform(dataset_manual)
df_manual = pd.DataFrame(te_ary, columns=te.columns_)

print("编码后的数据矩阵：")
display(df_manual)

编码后的数据矩阵：


,啤酒,尿布,花生
0,True,True,True
1,True,True,False
2,True,False,True
3,False,True,True
4,True,True,True
5,True,False,False


In [13]:
# 3. 挖掘频繁项集 (Support >= 0.5)
frequent_itemsets_manual = apriori(df_manual, min_support=0.5, use_colnames=True)

# 添加长度列方便观察
frequent_itemsets_manual['length'] = frequent_itemsets_manual['itemsets'].apply(lambda x: len(x))

print("频繁项集：")
display(frequent_itemsets_manual)

频繁项集：


,support,itemsets,length
0,0.833333,(啤酒),1
1,0.666667,(尿布),1
2,0.666667,(花生),1
3,0.500000,"(尿布, 啤酒)",2
4,0.500000,"(啤酒, 花生)",2
5,0.500000,"(尿布, 花生)",2


In [14]:
# 4. 生成关联规则
rules_manual = association_rules(frequent_itemsets_manual, metric="confidence", min_threshold=0.0)

# 选取关键列展示
cols = ['antecedents', 'consequents', 'support', 'confidence', 'lift']

print("生成的关联规则：")
display(rules_manual[cols])

生成的关联规则：


,antecedents,consequents,support,confidence,lift
0,(尿布),(啤酒),0.5,0.75,0.900
1,(啤酒),(尿布),0.5,0.60,0.900
2,(啤酒),(花生),0.5,0.60,0.900
3,(花生),(啤酒),0.5,0.75,0.900
4,(尿布),(花生),0.5,0.75,1.125
5,(花生),(尿布),0.5,0.75,1.125


### 结果对比

观察上表输出，与我们的手算结果一致：

- **{尿布} -> {啤酒}**：Confidence = 0.75
- **{啤酒} -> {尿布}**：Confidence = 0.60
- **{尿布} -> {花生}**：Confidence = 0.75
- **{花生} -> {尿布}**：Confidence = 0.75
- **{啤酒} -> {花生}**：Confidence = 0.60
- **{花生} -> {啤酒}**：Confidence = 0.75

## 3. 综合案例分析

接下来运行文档 **4. 工程实践** 中的完整案例。

In [15]:
# 1. 构建模拟交易数据
dataset = [
    ['牛奶', '尿布', '啤酒', '花生', '面包'],
    ['尿布', '啤酒', '火腿', '面包'],
    ['牛奶', '尿布', '啤酒', '可乐'],
    ['牛奶', '面包', '花生'],
    ['面包', '啤酒', '花生', '尿布'],
]

# 2. 数据编码
te = TransactionEncoder()
te_ary = te.fit(dataset).transform(dataset)
df = pd.DataFrame(te_ary, columns=te.columns_)

print("Encoded DataFrame Shape:", df.shape)
display(df.head())

Encoded DataFrame Shape: (5, 7)


,可乐,啤酒,尿布,火腿,牛奶,花生,面包
0,False,True,True,False,True,True,True
1,False,True,True,True,False,False,True
2,True,True,True,False,True,False,False
3,False,False,False,False,True,True,True
4,False,True,True,False,False,True,True


In [16]:
# 3. 挖掘频繁项集 (min_support=0.6)
frequent_itemsets = apriori(df, min_support=0.6, use_colnames=True)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(lambda x: len(x))

print("\n--- Frequent Itemsets (Top 5) ---")
display(frequent_itemsets.head())


--- Frequent Itemsets (Top 5) ---


,support,itemsets,length
0,0.8,(啤酒),1
1,0.8,(尿布),1
2,0.6,(牛奶),1
3,0.6,(花生),1
4,0.8,(面包),1


In [17]:
# 4. 生成强关联规则 (min_confidence=0.7)
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.7)

# 筛选高价值规则：提升度 > 1.0 且 置信度 > 0.7
strong_rules = rules[(rules['lift'] > 1.0) & (rules['confidence'] > 0.7)]

print("\n--- Strong Association Rules ---")
display(strong_rules[cols].sort_values(by='lift', ascending=False))


--- Strong Association Rules ---


,antecedents,consequents,support,confidence,lift
0,(尿布),(啤酒),0.8,1.00,1.25
1,(啤酒),(尿布),0.8,1.00,1.25
7,(花生),(面包),0.6,1.00,1.25
8,"(尿布, 面包)",(啤酒),0.6,1.00,1.25
10,"(面包, 啤酒)",(尿布),0.6,1.00,1.25
6,(面包),(花生),0.6,0.75,1.25
11,(尿布),"(面包, 啤酒)",0.6,0.75,1.25
13,(啤酒),"(尿布, 面包)",0.6,0.75,1.25


## 4. 性能优化：FP-Growth 算法

在大数据量场景下，使用 FP-Growth 算法可以获得更好的性能，且无需多次扫描数据库。

In [18]:
# 使用 FP-Growth 挖掘频繁项集
# 接口与 apriori 几乎一致
frequent_itemsets_fp = fpgrowth(df, min_support=0.6, use_colnames=True)

print("FP-Growth 结果 (前5条)：")
display(frequent_itemsets_fp.head())

FP-Growth 结果 (前5条)：


,support,itemsets
0,0.8,(面包)
1,0.8,(尿布)
2,0.8,(啤酒)
3,0.6,(花生)
4,0.6,(牛奶)
